# Capstone: Autonomous Survey Assistant

**WatSPEED Agentic AI prep — Week 6 - capstone**

Runs offline. Set `OPENAI_API_KEY` to swap the stub model for a real one.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() if (pathlib.Path.cwd() / 'agentkit.py').exists()
                      else pathlib.Path.cwd() / 'notebooks'))
from agentkit import *

## Capstone: an autonomous survey analysis assistant

The course capstone is "a personalized agentic AI assistant capable of autonomous task
execution." This notebook assembles every previous piece into one system, so you arrive
at the course with a working reference implementation rather than a blank file.

| Component | Notebook |
|---|---|
| Tools + validated arguments | 02 |
| Agent loop with a step budget | 03 |
| Context budget | 04 |
| RAG over the codebook + episodic memory | 05 |
| State machine with retry and approval gate | 06 |
| Tools served over MCP | 07 |
| Analyst / reviewer / writer roles | 08 |

In [2]:
# --- Knowledge base (notebook 05) ------------------------------------------
kb = VectorStore()
for text, src in [
    ("trusts_ai: trust in AI, 1=none to 5=complete. Missing coded -9.", "codebook"),
    ("age_group: 18-29, 30-44, 45-59, 60+.", "codebook"),
    ("All estimates must use the survey weight variable wt_final.", "methods"),
    ("Chi-square requires expected cell counts of at least 5.", "methods"),
]:
    kb.add(text, source=src)

# --- Tools (notebook 02) ----------------------------------------------------
tools = ToolBox()

@tools.tool("Look up a variable in the codebook", schema(variable='string'))
def lookup(variable: str) -> dict:
    hits = kb.search(variable, k=1)
    return {"variable": variable, "definition": hits[0][1] if hits else "not found"}

@tools.tool("Chi-square test between two survey variables",
            schema(row_var='string', col_var='string', weighted='string'))
def chi_square(row_var: str, col_var: str, weighted: str = "false") -> dict:
    return {"row": row_var, "col": col_var, "chi2": 12.41, "dof": 3,
            "p": 0.006, "n": 2041, "weighted": weighted == "true"}

print(tools.describe())

- lookup(variable): Look up a variable in the codebook
- chi_square(row_var, col_var, weighted): Chi-square test between two survey variables


### The assistant

In [3]:
class SurveyAssistant:
    def __init__(self, kb: VectorStore, tools: ToolBox, user: str):
        self.kb, self.tools, self.user = kb, tools, user
        self.memory = VectorStore()          # episodic, across runs
        self.audit: list[dict] = []          # every tool call, for reproducibility

    def _grounded_prompt(self, question: str) -> list[dict]:
        context = "\n".join(t for _, t, _ in self.kb.search(question, k=3))
        recalled = [t for _, t, _ in self.memory.search(question, k=2)]
        system = f"You analyse survey data.\n\nMethods notes:\n{context}"
        if recalled:
            system += "\n\nRemembered from earlier sessions:\n" + "\n".join(recalled)
        return [{"role": "system", "content": system}, {"role": "user", "content": question}]

    def analyse(self, llm, question: str, max_steps: int = 5) -> dict:
        messages = self._grounded_prompt(question)
        for step in range(1, max_steps + 1):
            resp = llm.chat(messages, tools=self.tools.schemas())
            if not resp.wants_tool:
                trace(step, "assistant", resp.content)
                return {"answer": resp.content, "audit": self.audit}
            call = resp.tool_calls[0]
            result = self.tools.call(call["name"], call["arguments"])
            self.audit.append({"step": step, **call, "result": result})
            trace(step, call["name"], str(result))
            messages.append({"role": "tool", "name": call["name"], "content": str(result)})
        return {"answer": "[hit step budget]", "audit": self.audit}

    def remember(self, fact: str) -> None:
        self.memory.add(fact, user=self.user)

### Run 1 — the reviewer catches an unweighted estimate

This is the whole system working: retrieval supplies the weighting rule, the agent loop
runs the test, the review node rejects it, the graph retries, and the gate asks a human.

In [4]:
assistant = SurveyAssistant(kb, tools, user="peter")

llm = get_llm([
    LLMResponse(tool_calls=[{"name": "lookup", "arguments": {"variable": "trusts_ai"}}]),
    LLMResponse(tool_calls=[{"name": "chi_square",
                             "arguments": {"row_var": "age_group", "col_var": "trusts_ai",
                                           "weighted": "false"}}]),
    LLMResponse(content="Trust differs by age group (chi2=12.41, p=0.006, N=2041)."),
])

banner("Run 1: unweighted")
run1 = assistant.analyse(llm, "Does trust in AI vary by age group?")

def review(audit: list[dict]) -> str:
    for entry in audit:
        if entry["name"] == "chi_square" and not entry["result"]["weighted"]:
            return "REJECT: estimate is unweighted; wt_final is required."
    return ""

verdict = review(run1["audit"])
print("\nReviewer:", verdict or "passed")

Using stub-llm (deterministic, offline) - no OPENAI_API_KEY found, so results are scripted.

Run 1: unweighted
  [ 1] lookup       | {'variable': 'trusts_ai', 'definition': 'trusts_ai: trust in AI, 1=none to 5=complete. Missing coded -9.'}
  [ 2] chi_square   | {'row': 'age_group', 'col': 'trusts_ai', 'chi2': 12.41, 'dof': 3, 'p': 0.006, 'n': 2041, 'weighted': False}
  [ 3] assistant    | Trust differs by age group (chi2=12.41, p=0.006, N=2041).

Reviewer: REJECT: estimate is unweighted; wt_final is required.


In [5]:
banner("Run 2: retry with weights applied")
llm2 = get_llm([
    LLMResponse(tool_calls=[{"name": "chi_square",
                             "arguments": {"row_var": "age_group", "col_var": "trusts_ai",
                                           "weighted": "true"}}]),
    LLMResponse(content="Weighted: trust rises with age (chi2=12.41, p=0.006, N=2041)."),
], verbose=False)

assistant.audit.clear()
run2 = assistant.analyse(llm2, "Re-run weighted: does trust in AI vary by age group?")
print("\nReviewer:", review(run2["audit"]) or "passed")

banner("Approval gate before anything leaves the system")
print("PAUSED - publish this finding to the shared report? [y/N]")
print("resume(approved=True) ->", "PUBLISHED:", run2["answer"])


Run 2: retry with weights applied
  [ 1] chi_square   | {'row': 'age_group', 'col': 'trusts_ai', 'chi2': 12.41, 'dof': 3, 'p': 0.006, 'n': 2041, 'weighted': True}
  [ 2] assistant    | Weighted: trust rises with age (chi2=12.41, p=0.006, N=2041).

Reviewer: passed

Approval gate before anything leaves the system
PAUSED - publish this finding to the shared report? [y/N]
resume(approved=True) -> PUBLISHED: Weighted: trust rises with age (chi2=12.41, p=0.006, N=2041).


### Audit trail — the thing that makes it defensible

In [6]:
show("Every tool call this run", run2["audit"])

assistant.remember("Peter requires weighted estimates using wt_final.")
assistant.remember("Peter analyses the AI trust survey, focusing on age effects.")
banner("Carried into the next session")
show("recall", [t for _, t, _ in assistant.memory.search("how should I compute means?", k=2)])

Every tool call this run:
  [
    {
      "step": 1,
      "name": "chi_square",
      "arguments": {
        "row_var": "age_group",
        "col_var": "trusts_ai",
        "weighted": "true"
      },
      "result": {
        "row": "age_group",
        "col": "trusts_ai",
        "chi2": 12.41,
        "dof": 3,
        "p": 0.006,
        "n": 2041,
        "weighted": true
      }
    }
  ]

Carried into the next session
recall:
  []


---
## Where you are

You can now write, from scratch: a tool schema, a validated function-call loop, a
context budget, a vector store and RAG prompt, a state machine with retry and a human
gate, an MCP server and client, and three multi-agent orchestration patterns.

The course will hand you LangChain, LangGraph, MCP and AutoGen. You will recognise every
one of them as a wrapper around something in these notebooks — which is the difference
between using a framework and debugging one.

### Before the course starts (Oct 5, 2026)

1. Create a venv and install `requirements.txt`; re-run notebooks 02-09 for real.
2. Get an API key and set `OPENAI_API_KEY`, then re-run. Every notebook switches over
   with no code change — watch where the stub's determinism was hiding real variance.
3. Port one notebook to the real framework: 03 to `create_react_agent`, or 06 to `StateGraph`.
4. Swap the capstone's fake `chi_square` for real `pandas` + `scipy` against
   `data/ai_trust_insights.csv`.